# Trying society emulations

#### Referring to these projects:

[OASIS](https://github.com/camel-ai/oasis),
[MiroFish-Offline](https://github.com/nikmcfly/MiroFish-Offline),
[miroclaw](https://github.com/justinfinnn/miroclaw).

Those run thousands of agents on a graph database and a GPU. This is the same idea
at the size of one laptop: personas react to a post and to each other, before it goes
out. A simulation only talks in memory, no plugin is involved, so nothing it says can
reach a platform.

In [1]:
import os

from amoorion import Amoorion

# A sandbox for this whole notebook: secrets only from variables, the inbox and
# the dataset in a temporary folder. Nothing here can touch your real setup.
Amoorion.init()

PosixPath('/tmp/amoorion-05ezbezh')

In [2]:
amo = Amoorion()

# People made up for the occasion; `stance` decides how they answer.
personas = [
    {
        "name": "ada",
        "bio": "Follows every release.",
        "stance": "supportive",
    },
    {
        "name": "bo",
        "bio": "Has been burned before.",
        "stance": "critical",
    },
    {
        "name": "cy",
        "bio": "Just arrived.",
        "stance": "neutral",
    },
    {
        "name": "di",
        "bio": "Reads, never writes.",
        "stance": "silent",
    },
]

# Without a model, by simple scripts. Nothing reaches any platform.
the_matrix_movie = amo.simulate(
    topic="moving to our own server",
    personas=personas,
    rounds=2,
    run="trial",
)

# The feed: they react to the topic and to each other.
for message in the_matrix_movie:
    print(
        f"{message['author']['handle']:4}",
        message["text"],
    )

ada  moving to our own server? Yes, finally.
bo   @ada I doubt moving to our own server is what is needed.
cy   @bo What does moving to our own server change, in practice?
ada  @cy moving to our own server? Yes, finally.
bo   @ada I doubt moving to our own server is what is needed.
cy   @bo What does moving to our own server change, in practice?


Without a model these lines are scripted by stance: the same every time. They show
the mechanics and predict nothing. What was said stays in the inbox like anything
else, so rules and searches work on it:

In [3]:
# It went into the inbox, under its own name, so it can be searched.
[
    message["text"]
    for message in amo.search(text="doubt", platform="society:trial")
][:1]

['@ada I doubt moving to our own server is what is needed.']

With `llm=True` each persona asks a language model. Here that is a fake server:
point `llm/url` in the vault at Ollama, and the words become real.

In [4]:
# A small fake of a language model, as in notebook 11.
import json, threading

from http.server import BaseHTTPRequestHandler, HTTPServer


class FakeModel(BaseHTTPRequestHandler):
    """Speaks the chat completions API, as Ollama or a LiteLLM proxy would."""

    def do_POST(self):

        asked = json.loads(self.rfile.read(int(self.headers["Content-Length"])))

        said = asked["messages"][-1]["content"].splitlines()[0][:50]

        answer = {
            "choices": [
                {"message": {"content": f"(a model's words about: {said})"}}
            ]
        }

        self.send_response(200)

        self.end_headers()

        self.wfile.write(json.dumps(answer).encode())

    def log_message(
        self,
        *args,
    ):

        pass


# Port 0: the system picks a free one.
model_server = HTTPServer(
    (
        "127.0.0.1",
        0,
    ),
    FakeModel,
)

threading.Thread(
    target=model_server.serve_forever,
    daemon=True,
).start()

# Where amoorion finds the model.
os.environ["AMOORION_SECRET_LLM_URL"] = (
    f"http://127.0.0.1:{model_server.server_port}/v1"
)

os.environ["AMOORION_SECRET_LLM_MODEL"] = "fake"

In [5]:
# `llm=True` lets the model speak for each persona.
ultimate_dating_simulator = amo.simulate(
    topic="moving to our own server",
    personas=personas[:2],
    rounds=1,
    run="by-model",
    llm=True,
)

for message in ultimate_dating_simulator:
    print(
        f"{message['author']['handle']:4}",
        message["text"],
    )

ada  (a model's words about: The topic: moving to our own server)
bo   (a model's words about: The topic: moving to our own server)


In Python, anything callable can think, a model of your own included:

In [ ]:
# The society by itself, without the method.
from amoorion.society import Persona, Society


# Your own way of thinking: `None` means staying silent.
def shout(
    persona,
    topic,
    feed,
):

    return f"{topic.upper()}!" if len(feed) < 3 else None


# Any function will do, so anything can be simulated.
[
    message.text
    for message in Society(
        [Persona(name="e"), Persona(name="f")], think=shout
    ).simulate("news")
]

['NEWS!', 'NEWS!', 'NEWS!']

# emotional drivers can be some of below mentioned

```
W przeciwieństwie do generatywnych modeli AI (jak ChatGPT), które „wymyślają” tekst, te prostsze modele są analityczne i klasyfikacyjne – dostają tekst i przypisują mu konkretną kategorię, etykietę lub wartość liczbową.
Oto zestawienie najważniejszych i najpopularniejszych prostych modeli podzielonych na kategorie:

Klasyfikacja i Filtrowanie Tekstu

• Analiza sentymentu (Sentiment Analysis): Ocenia emocjonalny ton wypowiedzi. Najczęściej dzieli tekst na trzy kategorie: pozytywny, negatywny lub neutralny.
• Detekcja toksyczności i oflagowanych treści (Content Moderation / Toxicity Detection): Wykrywa mowę nienawiści, wulgaryzmy, przemoc czy spam. Służy do automatycznego moderowania komentarzy.
• Klasyfikacja intencji (Intent Classification): Rozpoznaje, czego chce użytkownik (np. w chatbotach bankowych decyduje, czy klient chce „sprawdzić saldo”, czy „zablokować kartę”).

Wyciąganie Informacji i Semantyka

• Rozpoznawanie encji nazwanych (NER - Named Entity Recognition): Wyłapuje z tekstu konkretne obiekty i przypisuje im kategorie, np. imiona ludzi, nazwy firm, daty, kwoty czy lokalizacje.
• Analiza semantyczna i podobieństwo tekstów (Semantic Similarity): Mierzy, jak bliskie znaczeniowo są dwa zdania (nawet jeśli używają zupełnie innych słów, np. „Jak naprawić koło?” i „Instrukcja zmiany opony”).
• Wykrywanie języka (Language Detection): Natychmiastowo rozpoznaje, w jakim języku (np. polskim, angielskim, hiszpańskim) został napisany dany tekst.

Tradycyjne Algorytmy (Co stoi pod maską?)

Te prostsze zadania bardzo często realizuje się za pomocą mniejszych, starszych architektur, które nie potrzebują potężnych kart graficznych:
• Modele typu BERT (np. RoBERTa, DistilBERT): Małe modele transformerowe, które świetnie "rozumieją" kontekst i są mistrzami w klasyfikacji oraz NER.
• Algorytmy statystyczne (TF-IDF, Naive Bayes, SVM): Klasyczne metody matematyczne. Nie wymagają sieci neuronowych, są błyskawiczne i idealnie sprawdzają się np. w filtrach antyspamowych.
Czy potrzebujesz tych modeli do konkretnego projektu programistycznego, czy chcesz po prostu porównać ich skuteczność i koszty z nowoczesnymi modelami generatywnymi?

```